## Imports

In [2]:
#cell 1    #to do: update 2023,2024 function with headers , document headers problem 
import requests
import csv
import os
import re
import io
import calendar
from PyPDF2 import PdfReader

## File Paths, Downloading PDFs, and Helper Functions for Data Extraction

In [9]:
# cell 2 

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

DATE_PATTERN = re.compile(r"^(\d{2})-(\d{2})-(\d{4})(?:_\d+)?$", re.IGNORECASE)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,text/html,*/*",
    "Referer": "https://www.lawpd.com/",
}

def download_pdfs(
    start_id=15221,
    end_id=15239,
    base_url="https://lawpd.com/DocumentCenter/View/{}",
    failure_csv="failures.csv"
):
    """
    Download PDFs by incrementing through IDs, skipping failures,
    organizing them by date if the filename is in MM-DD-YYYY format,
    or else placing them in a fallback 'no_date' folder with a
    best-effort headline-based filename. Logs errors to a CSV file.
    """

    os.makedirs(DATA_DIR, exist_ok=True)

    failure_csv_path = os.path.join(DATA_DIR, failure_csv)

    total = end_id - start_id + 1

    with open(failure_csv_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["ID", "Error"])

        for count, file_id in enumerate(range(start_id, end_id + 1), start=1):
            if count % 500 == 0:
                print(f"📦 Progress: {count} / {total} checked")

            url = base_url.format(file_id)

            try:
                response = requests.get(
                    url,
                    headers=HEADERS,
                    timeout=20,
                    allow_redirects=True
                )

                status_code = response.status_code

                if status_code != 200:
                    writer.writerow([file_id, f"HTTP status {status_code}"])
                    continue

                content_type = response.headers.get("Content-Type", "").lower()
                if "pdf" not in content_type:
                    writer.writerow([file_id, f"Not a PDF (content-type: {content_type})"])
                    continue

                content_disp = response.headers.get("Content-Disposition", "")
                server_filename = get_filename_from_content_disposition(content_disp)

                if not server_filename:
                    server_filename = f"{file_id}.pdf"

                if not server_filename.lower().endswith(".pdf"):
                    server_filename += ".pdf"
                server_filename = re.sub(
                    r"_(\d+)(\.pdf)$",
                    r"\2",
                    server_filename,
                    flags=re.IGNORECASE
                    )
                    

                year, month, day = parse_date_from_filename(server_filename)

                if year is not None:
                    year_folder = f"{year}_law_pd_data"
                    month_name = calendar.month_name[month].lower()
                    month_folder = f"{year}_{month_name}"

                    year_folder_path = os.path.join(DATA_DIR, year_folder)
                    month_folder_path = os.path.join(year_folder_path, month_folder)

                    os.makedirs(month_folder_path, exist_ok=True)

                    file_path = os.path.join(month_folder_path, server_filename)

                else:
                    pdf_bytes = response.content
                    headline = extract_pdf_headline(pdf_bytes)

                    if headline:
                        safe_headline = sanitize_filename(headline)
                        fallback_name = f"{file_id}_{safe_headline}.pdf"
                    else:
                        fallback_name = f"{file_id}_no_headline.pdf"

                    fallback_folder_path = os.path.join(DATA_DIR, "no_date")
                    os.makedirs(fallback_folder_path, exist_ok=True)

                    file_path = os.path.join(fallback_folder_path, fallback_name)

                with open(file_path, "wb") as out_file:
                    out_file.write(response.content)

                print(f"[{count}/{total}] ✅ Saved: {server_filename}")

            except Exception as e:
                writer.writerow([file_id, str(e)])
                continue


def get_filename_from_content_disposition(content_disp):
    if "filename=" in content_disp.lower():
        parts = content_disp.split("filename=")
        if len(parts) > 1:
            filename_part = parts[1].strip().strip('"').strip(';')
            return filename_part
    return None


def parse_date_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = DATE_PATTERN.match(base_name)

    if not match:
        return (None, None, None)

    mm = int(match.group(1))
    dd = int(match.group(2))
    yyyy = int(match.group(3))

    if 1 <= mm <= 12 and 1 <= dd <= 31:
        return (yyyy, mm, dd)
    else:
        return (None, None, None)


def extract_pdf_headline(pdf_bytes):
    try:
        pdf_stream = io.BytesIO(pdf_bytes)
        pdf_reader = PdfReader(pdf_stream)

        if len(pdf_reader.pages) > 0:
            first_page = pdf_reader.pages[0]
            text = first_page.extract_text() or ""
            lines = text.splitlines()

            if lines:
                return lines[0].strip()

        return None

    except:
        return None


def sanitize_filename(name):
    safe = re.sub(r"[^a-zA-Z0-9_\-]+", "_", name)
    return safe[:50]

In [4]:
march_dir = os.path.join(DATA_DIR, "2019_law_pd_data", "2019_march")

for filename in sorted(os.listdir(march_dir)):
    if filename.startswith("03-") and "2019" in filename:
        print(filename)

03-01-2019_1.pdf
03-02-2019_1.pdf
03-03-2019_1.pdf
03-04-2019_1.pdf
03-05-2019_1.pdf
03-06-2019_2.pdf
03-07-2019_1.pdf
03-08-2019_1.pdf
03-09-2019_1.pdf
03-10-2019_1.pdf
03-11-2019_1.pdf
03-12-2019_1.pdf
03-13-2019_1.pdf
03-14-2019.pdf


## Main Call

In [10]:
download_pdfs()

[1/19] ✅ Saved: 06-15-2019.pdf
[2/19] ✅ Saved: 06-16-2019.pdf
[3/19] ✅ Saved: 06-17-2019.pdf
[4/19] ✅ Saved: 06-18-2019.pdf
[5/19] ✅ Saved: 06-19-2019.pdf
[6/19] ✅ Saved: 06-01-2019.pdf
[7/19] ✅ Saved: 06-02-2019.pdf
[8/19] ✅ Saved: 06-03-2019.pdf
[9/19] ✅ Saved: 06-04-2019.pdf
[10/19] ✅ Saved: 06-05-2019.pdf
[11/19] ✅ Saved: 06-06-2019.pdf
[12/19] ✅ Saved: 06-07-2019.pdf
[13/19] ✅ Saved: 06-08-2019.pdf
[14/19] ✅ Saved: 06-09-2019.pdf
[15/19] ✅ Saved: 06-10-2019.pdf
[16/19] ✅ Saved: 06-11-2019.pdf
[17/19] ✅ Saved: 06-12-2019.pdf
[18/19] ✅ Saved: 06-13-2019.pdf
[19/19] ✅ Saved: 06-14-2019.pdf


## Operation 2023, 2024

In [1]:
import os
import requests
import re
import calendar
from datetime import datetime

# Base data path using current notebook directory
notebook_dir = os.getcwd()

DATE_PATTERN = re.compile(r"^(\d{2})-(\d{2})-(\d{4})$", re.IGNORECASE)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

# Browser-like headers to avoid the Lawrence site blocking Python requests
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,text/html,*/*",
    "Referer": "https://www.lawpd.com/",
}

def get_filename_from_content_disposition(header):
    if "filename=" in header.lower():
        return header.split("filename=")[-1].strip('"; ')
    return None

def parse_date_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = DATE_PATTERN.match(base_name)

    if match:
        mm, dd, yyyy = int(match.group(1)), int(match.group(2)), int(match.group(3))
        return yyyy, mm, dd

    return None, None, None

def download_pdfs_two(start_id=45274, end_id=65274):
    os.makedirs(DATA_DIR, exist_ok=True)

    total = end_id - start_id + 1

    skipped = {
        "http": 0,
        "not_pdf": 0,
        "missing_filename": 0,
        "wrong_year": 0,
        "error": 0
    }

    saved = 0

    for count, doc_id in enumerate(range(start_id, end_id + 1), start=1):
        if count % 500 == 0:
            print(f"📦 Progress: {count} / {total} checked")

        url = f"https://www.lawpd.com/DocumentCenter/View/{doc_id}"

        try:
            res = requests.get(
                url,
                headers=HEADERS,
                timeout=20,
                allow_redirects=True
            )

            if res.status_code != 200:
                skipped["http"] += 1
                continue

            content_type = res.headers.get("Content-Type", "").lower()
            if "pdf" not in content_type:
                skipped["not_pdf"] += 1
                continue

            filename = get_filename_from_content_disposition(
                res.headers.get("Content-Disposition", "")
            )

            if not filename:
                skipped["missing_filename"] += 1
                continue

            if not filename.lower().endswith(".pdf"):
                filename += ".pdf"

            year, month, day = parse_date_from_filename(filename)

            if year is None:
                skipped["wrong_year"] += 1
                continue

            # Build folder path: .../data/2023_january
            month_folder = f"{year}_{calendar.month_name[month].lower()}"
            save_dir = os.path.join(DATA_DIR, month_folder)
            os.makedirs(save_dir, exist_ok=True)

            save_path = os.path.join(save_dir, filename)

            with open(save_path, "wb") as f:
                f.write(res.content)

            saved += 1
            print(f"[{count}/{total}] ✅ Saved: {filename}")

        except Exception as e:
            skipped["error"] += 1

            if skipped["error"] <= 5:
                print(f"{doc_id}: {type(e).__name__}: {e}")

    print(f"Done. Saved {saved} PDFs to {DATA_DIR}")
    print(f"Skipped: {skipped}")

In [6]:
import socket, sys
print(sys.executable)
print(socket.getaddrinfo("lawpd.com", 443)[0])


/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/venv/bin/python
(<AddressFamily.AF_INET: 2>, <SocketKind.SOCK_DGRAM: 2>, 17, '', ('207.38.72.200', 443))


In [7]:
import requests
r = requests.get("https://lawpd.com/DocumentCenter/View/42774", timeout=10)
print(r.status_code, r.url, r.headers.get("content-type"))


404 https://lawpd.com/DocumentCenter/View/42774 None


In [16]:
download_pdfs_two()

📦 Progress: 500 / 20001 checked
[966/20001] ✅ Saved: 01-01-2023.pdf
[972/20001] ✅ Saved: 01-02-2023.pdf
[973/20001] ✅ Saved: 01-03-2023.pdf
[977/20001] ✅ Saved: 01-04-2023.pdf
[978/20001] ✅ Saved: 01-05-2023.pdf
[981/20001] ✅ Saved: 01-06-2023.pdf
[985/20001] ✅ Saved: 01-07-2023.pdf
[988/20001] ✅ Saved: 01-08-2023.pdf
[989/20001] ✅ Saved: 01-09-2023.pdf
[997/20001] ✅ Saved: 01-10-2023.pdf
📦 Progress: 1000 / 20001 checked
[1001/20001] ✅ Saved: 01-11-2023.pdf
[1010/20001] ✅ Saved: 01-12-2023.pdf
[1016/20001] ✅ Saved: 01-13-2023.pdf
[1017/20001] ✅ Saved: 01-14-2023.pdf
[1019/20001] ✅ Saved: 01-16-2023.pdf
[1020/20001] ✅ Saved: 01-15-2023.pdf
[1031/20001] ✅ Saved: 01-17-2023.pdf
[1037/20001] ✅ Saved: 01-18-2023.pdf
[1038/20001] ✅ Saved: 01-19-2023.pdf
[1050/20001] ✅ Saved: 01-20-2023.pdf
[1054/20001] ✅ Saved: 01-22-2023.pdf
[1055/20001] ✅ Saved: 01-23-2023.pdf
[1060/20001] ✅ Saved: 01-24-2023.pdf
[1067/20001] ✅ Saved: 01-25-2023.pdf
[1072/20001] ✅ Saved: 01-26-2023.pdf
[1073/20001] ✅ Saved